# Continent Strength Profile

This example uses `gdmate.rheology_models.flow_laws` to convert experimental
coefficients and calculate an illustrative continental strength profile.
It compares wet quartzite in the upper crust and wet anorthite in the lower
crust using the ASPECT continental extension parameterization. Pressure and
temperature profiles are illustrative and can be replaced with measured or
modeled profiles.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from gdmate.rheology_models.flow_laws import (
    GLEASON_TULLIS_1995,
    RYBACKI_2006,
    to_aspect,
)
from gdmate.rheology_models.rheology import cond_geotherm


## Experimental laws and prefactor conversion

Record the experimental law as
$$
\dot\epsilon_{\rm exp}=A_{\rm exp}\sigma_{\rm exp}^{n}
 d_{\rm exp}^{-p} f_{\rm exp}^{r}
 \exp\left[-\frac{E+PV}{RT}\right].
$$
Here the subscript $\rm exp$ denotes quantities expressed in the experimental
units and stress/strain-rate convention:

* $\dot\epsilon_{\rm exp}$ is the experimental strain rate, in $\mathrm{s}^{-1}$.
  Depending on the recorded convention, it is the axial strain rate,
  engineering shear strain rate, or strain-rate second invariant.
* $A_{\rm exp}$ is the reported experimental prefactor. Its units are
  $(\text{stress unit})^{-n}(\text{grain-size unit})^p
  (\text{fugacity unit})^{-r}\,\mathrm{s}^{-1}$.
* $\sigma_{\rm exp}$ is the stress in the experimental stress unit, such as
  MPa. It represents differential stress for a uniaxial law, shear stress
  for an engineering-shear law, or the stress invariant for an invariant law.
* $d_{\rm exp}$ is grain size in the experimental length unit, such as
  micrometres. The exponent $p$ describes grain-size dependence through
  $d_{\rm exp}^{-p}$; $p=0$ means no grain-size dependence.
* $f_{\rm exp}$ is water fugacity in the experimental fugacity unit, such as
  MPa. The exponent $r$ describes explicit water dependence through
  $f_{\rm exp}^{r}$; $r=0$ means no explicit fugacity dependence.
* $n$ is the stress exponent: at fixed grain size, fugacity, pressure and
  temperature, strain rate is proportional to $\sigma_{\rm exp}^n$.
* $E$ is activation energy, in $\mathrm{J\,mol}^{-1}$.
* $P$ is pressure, always in Pa, even if $\sigma_{\rm exp}$ uses MPa.
* $V$ is activation volume, in $\mathrm{m}^3\,\mathrm{mol}^{-1}$, so $PV$
  has the same units as $E$.
* $R=8.314462618\ldots\ \mathrm{J\,mol}^{-1}\,\mathrm{K}^{-1}$ is the
  molar gas constant, and $T$ is absolute temperature in K.
* $\exp[-(E+PV)/(RT)]$ is the dimensionless Arrhenius factor describing
  thermal activation and the pressure contribution to the activation energy.

Units of stress, grain size and fugacity are attached to the record. Here E is
always J/mol and V is m³/mol. The converted prefactor is
$$
A_{\rm ASPECT}=A_{\rm exp} U_\sigma^{-n}U_d^p U_f^{-r}
 F_s f_{\rm Pa}^r,
$$
Here:

* $A_{\rm ASPECT}$ is the converted prefactor, in
  $\mathrm{Pa}^{-n}\,\mathrm{m}^{p}\,\mathrm{s}^{-1}$, with the specified
  water fugacity incorporated into it.
* $U_\sigma$, $U_d$, and $U_f$ are the numerical conversion factors from one
  experimental unit of stress, grain size, and fugacity to its SI unit.
  Thus $\sigma_{\rm Pa}=U_\sigma\sigma_{\rm exp}$,
  $d_{\rm m}=U_d d_{\rm exp}$, and $f_{\rm Pa}=U_f f_{\rm exp}$.
* $f_{\rm Pa}$ is the prescribed water fugacity expressed in Pa. Combining
  $U_f^{-r}f_{\rm Pa}^r=(f_{\rm Pa}/U_f)^r$ evaluates the water term in
  the experiment's fugacity units. For $r=0$, this factor is one; no fugacity
  value is required. Water content is not a substitute for water fugacity.
* $F_s$ converts the experimental stress/strain-rate convention to ASPECT's
  strain-rate invariant $\dot\epsilon_{II}$ and deviatoric stress invariant
  $\tau_{II}$. Both invariants are defined as
  $\sqrt{\tfrac12\mathbf{X}:\mathbf{X}}$ for the corresponding tensor.

For MPa and micrometres, $U_\sigma=10^6$ and $U_d=10^{-6}$, so the
stress/grain-size factor is
$$
U_\sigma^{-n}U_d^p=(10^6)^{-n}(10^{-6})^p=10^{-6n-6p}.
$$
If fugacity is also reported in MPa, $U_f=10^6$. For example, at
$f_{\rm Pa}=10^6$ Pa (1 MPa), $U_f^{-r}f_{\rm Pa}^r=1$.

For **uniaxial axial strain rate versus differential stress**, the
strain-rate-convention conversion is
$$
F_s=\frac{3^{(n+1)/2}}{2},
$$
because $\dot\epsilon_{II}=\sqrt{3}\,\dot\epsilon_{\rm axial}/2$ and
$\tau_{II}=\sigma_{\rm differential}/\sqrt{3}$ for incompressible uniaxial
viscous deformation.

For **engineering shear strain rate** $\dot\gamma$ versus shear stress,
$F_s=1/2$, because $\dot\epsilon_{II}=\dot\gamma/2$ and
$\tau_{II}=\tau_{\rm shear}$. If the experimental law already uses the
same second-invariant definitions as ASPECT, $F_s=1$.

## In this notebook, the flow laws for crustal materials are:

* Upper crust: Gleason–Tullis (1995), $A_{\rm exp}=1.1\times10^{-4}$,
  $n=4$, $E=223$ kJ/mol, $V=0$. Wet calibration, **no explicit fugacity term**.
* Lower crust: Rybacki et al. (2006), cookbook parameterization,
  $A_{\rm exp}=10^{0.2}$, $n=3$, $E=345$ kJ/mol, $V=0$, $r=1$.
  The cookbook assumes $f_{\rm H_2O}=1$ MPa.

References: [Gleason and Tullis (1995)](https://doi.org/10.1016/0040-1951(95)00011-B),
[Rybacki et al. (2006)](https://doi.org/10.1029/2005JB003663), and
[Dannberg et al. (2017), SI S1.3](https://doi.org/10.1002/2017GC006944).
Conversion follows ASPECT's `cookbooks/continental_extension/continental_extension_rheology.py`.


In [ ]:
# Choose water fugacity and the strain-rate invariant for this example.
lower_crust_water_fugacity_mpa = 1.0
strain_rate = 1e-15  # second invariant, s^-1

upper_conversion = to_aspect(GLEASON_TULLIS_1995)
lower_conversion = to_aspect(
    RYBACKI_2006, water_fugacity_pa=lower_crust_water_fugacity_mpa * 1e6
)
upper = upper_conversion.law
lower = lower_conversion.law

print("Upper crust:", upper)
print("Lower crust:", lower)
for conversion in (upper_conversion, lower_conversion):
    print(conversion.experimental.name)
    print("  unit factor:", conversion.unit_factor)
    print("  invariant factor:", conversion.strain_rate_factor)
    print("  water factor (Pa units):", conversion.water_factor)

assert np.isclose(upper.A, 8.57e-28, rtol=5e-4, atol=0)
if lower_crust_water_fugacity_mpa == 1.0:
    assert np.isclose(lower.A, 7.13e-18, rtol=5e-4, atol=0)


### Water dependence and derived coefficients

The quartzite law has no explicit water-fugacity dependence. Water content
cannot be substituted for fugacity. The anorthite law has fugacity exponent
r=1, so changing the specified fugacity changes its converted prefactor.

Conversions retain full precision. At 1 MPa water fugacity for anorthite,
the derived ASPECT prefactors are approximately `8.57e-28` for quartzite and
`7.13e-18` for anorthite. The conversion accounts for units, strain-rate
conventions and explicit water dependence.


## Construct an illustrative strength profile

Upper/lower crust boundaries are 20/40 km. A conductive Chapman-style geotherm
uses gdmate's existing utility. Pressure is integrated with layer densities
2700/2900/3300 kg/m³ and gravity 9.81 m/s². Replace these arrays with a case's
pressure and temperature output for a model-specific comparison.

ASPECT dislocation viscosity is
$$
\eta=\tfrac12 A^{-1/n}d^{p/n}\dot\epsilon_{II}^{(1-n)/n}
\exp\left[\frac{E+PV}{nRT}\right].
$$
Both crustal laws have p=0. This example includes dislocation creep and
plastic yielding; diffusion and Peierls creep are omitted. For 2-D Drucker–Prager yielding the stress limit is
$\tau_y=C\cos\phi+P\sin\phi$, with $C=20$ MPa and $\phi=30^\circ$.
The composite viscosity is the minimum of viscous and plastic viscosities.
No strain weakening, phase changes or viscosity bounds are included.


In [ ]:
_, _, depth_m, temperature_k = cond_geotherm(
    thicknesses=[20, 20, 60], depth=100,
    radiogenic_heat=[1e-6, 2.5e-7, 0.0],
    surface_t=273.0, heat_flow=0.055, thermal_conductivity=2.5,
)
pressure_pa = 9.81 * (
    2700 * np.minimum(depth_m, 20e3)
    + 2900 * np.clip(depth_m - 20e3, 0, 20e3)
    + 3300 * np.maximum(depth_m - 40e3, 0)
)

# Only plot the crust: the mantle has a separate flow law.
crust = depth_m <= 40e3
z = depth_m[crust]
t = temperature_k[crust]
p = pressure_pa[crust]
eta_upper = upper.viscosity(strain_rate, t, p)
eta_lower = lower.viscosity(strain_rate, t, p)
eta_viscous = np.where(z <= 20e3, eta_upper, eta_lower)
cohesion_pa = 20e6
friction_angle_deg = 30.0
phi = np.deg2rad(friction_angle_deg)
yield_stress_pa = cohesion_pa * np.cos(phi) + p * np.sin(phi)
eta_plastic = yield_stress_pa / (2 * strain_rate)
eta_composite = np.minimum(eta_viscous, eta_plastic)

assert np.all(np.isfinite(eta_composite))
assert np.all(eta_composite > 0)
assert np.all(2 * strain_rate * eta_composite <= yield_stress_pa * (1 + 1e-12))

fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharey=True)
axes[0].plot(t - 273.15, z / 1e3)
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Depth (km)")
axes[1].plot(2 * strain_rate * eta_composite / 1e6, z / 1e3, label="Composite")
axes[1].plot(yield_stress_pa / 1e6, z / 1e3, "--", label="Plastic limit")
axes[1].set_xlabel("Stress invariant (MPa)")
axes[1].legend()
axes[2].plot(np.log10(eta_viscous), z / 1e3, label="Dislocation")
axes[2].plot(np.log10(eta_composite), z / 1e3, label="Composite")
axes[2].set_xlabel("log10 viscosity (Pa s)")
axes[2].legend()
for ax in axes:
    ax.axhline(20, color="gray", linestyle=":")
    ax.grid(alpha=0.3)
axes[0].invert_yaxis()
fig.tight_layout()
plt.show()
